# AICE-Style Practice: Seoul Food Delivery (Regression)
이 문제세트는 공개 AICE 샘플과 **유사한 흐름**으로 구성되었지만, **완전히 새로운 데이터셋**을 사용합니다.

### 데이터
- `food_delivery_seoul.csv` : 주문 단위의 배달 기록
- `traffic_events.csv` : 동일 `OrderID` 기준의 교차 신호등 수

### 목표(Target)
- `Delivery_Time_Minutes` (분) 예측

### 권장 절차(문항 1~14)
임포트 → 읽기/머지 → 시각화(countplot, jointplot) → 이상치 제거 → 결측 처리 → 불필요 컬럼 삭제 → 원-핫 인코딩 → 스케일링 → 의사결정나무/랜덤포레스트 → MAE 비교 → 간단 MLP 회귀 → 학습곡선 시각화


> **해설 노트**: 아래 코드는 위 템플릿 각 문항의 모범 풀이입니다.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
plt.rc('font', family='NanumGothicCoding')

In [ ]:
df_main = pd.read_csv('food_delivery_seoul.csv')
df_traffic = pd.read_csv('traffic_events.csv')
df = pd.merge(df_main, df_traffic, on='OrderID', how='inner')
df.head()

In [ ]:
sns.countplot(data=df, x='Region1'); plt.xticks(rotation=25); plt.show()
df = df[df['Region1']!='-'].copy()
# 보기 구성에 따라 달라질 수 있으나, 분포상 '서울특별시'가 가장 크도록 생성됨
답안04 = 3

In [ ]:
sns.jointplot(data=df, x='Distance_km', y='Delivery_Time_Minutes')

In [ ]:
df_temp = df[df['Avg_Speed_kmh'] < 120].copy()
df_temp.drop(columns=['OrderID'], inplace=True)
na_cnt = int(df_temp.isnull().sum().sum())
df_na = df_temp.dropna().copy()
답안07 = na_cnt; 답안07

In [ ]:
df_del = df_na.drop(columns=['Time_Order_Placed','Time_Order_Delivered'], errors='ignore').copy()
obj_cols = df_del.select_dtypes(include='object').columns.tolist()
df_preset = pd.get_dummies(df_del, columns=obj_cols)
X = df_preset.drop('Delivery_Time_Minutes', axis=1).values
y = df_preset['Delivery_Time_Minutes'].values
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.preprocessing import RobustScaler
rs = RobustScaler(); X_train = rs.fit_transform(X_train); X_valid = rs.transform(X_valid)

In [ ]:
dt = DecisionTreeRegressor(max_depth=6, min_samples_split=4, random_state=120)
rf = RandomForestRegressor(max_depth=6, min_samples_split=4, random_state=120)
dt.fit(X_train, y_train); rf.fit(X_train, y_train)
y_pred_dt = dt.predict(X_valid); y_pred_rf = rf.predict(X_valid)
dt_mae = mean_absolute_error(y_valid, y_pred_dt)
rf_mae = mean_absolute_error(y_valid, y_pred_rf)
답안12 = 'randomforest' if rf_mae < dt_mae else 'decisiontree'
dt_mae, rf_mae, 답안12

In [ ]:
tf.random.set_seed(7)
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'), Dropout(0.2),
    Dense(16, activation='relu'), Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mse'])
history = model.fit(X_train, y_train, epochs=25, batch_size=32, validation_data=(X_valid, y_valid), verbose=0)
plt.plot(history.history['mse']); plt.plot(history.history['val_mse']);
plt.title('Model MSE'); plt.xlabel('Epochs'); plt.ylabel('MSE'); plt.legend(['mse','val_mse']); plt.show()